# Домашнее задание 1.2: Баллистическое движение в 2D

В данном ноутбуке выполнено численное моделирование полета тела под действием гравитации с учетом сопротивления среды. Реализован анализ точности методов интегрирования ОДУ (Эйлера и Рунге-Кутты 4-го порядка), а также исследовано влияние линейного (🟢) и квадратичного (🟡) сопротивления на характеристики траектории.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# Общие физические параметры
g = 9.81
v0 = 50.0

# Численные параметры
dt = 0.05
t_max = 15.0
n_steps = int(t_max / dt)

def explicit_euler_step(s, dt, d_func):
    """Явный метод Эйлера"""
    return s + dt * d_func(s)

def rk4_step(s, dt, d_func):
    """Метод Рунге-Кутты 4-го порядка"""
    d1 = dt * d_func(s)
    d2 = dt * d_func(s + d1 / 2)
    d3 = dt * d_func(s + d2 / 2)
    d4 = dt * d_func(s + d3)
    return s + (d1 + 2 * d2 + 2 * d3 + d4) / 6

## 🟢 Базовый уровень: Линейное сопротивление среды

Рассматривается задача Коши для системы обыкновенных дифференциальных уравнений:
$$ \frac{dx}{dt} = v_x, \quad \frac{dy}{dt} = v_y $$
$$ \frac{dv_x}{dt} = -\frac{v_x}{\tau}, \quad \frac{dv_y}{dt} = -g - \frac{v_y}{\tau} $$

Точное аналитическое решение:
$$ x(t) = v_{x0} \tau (1 - e^{-t/\tau}) $$
$$ y(t) = \tau (v_{y0} + g\tau)(1 - e^{-t/\tau}) - g\tau t $$

При наличии диссипативной силы (сопротивления) полная механическая энергия системы не сохраняется, а переходит во внутреннюю энергию.

In [ ]:
alpha = np.radians(45.0)
tau = 5.0  # Характерное время торможения
m = 1.0    # Масса тела

vx0 = v0 * np.cos(alpha)
vy0 = v0 * np.sin(alpha)

def d_linear(s):
    x, y, vx, vy = s
    return np.array([vx, vy, -vx / tau, -g - vy / tau])

def analytical_solution(t):
    x = vx0 * tau * (1 - np.exp(-t / tau))
    y = tau * (vy0 + g * tau) * (1 - np.exp(-t / tau)) - g * tau * t
    return x, y

s_euler = np.zeros((n_steps + 1, 4))
s_rk4 = np.zeros((n_steps + 1, 4))
s_0 = np.array([0.0, 0.0, vx0, vy0])

s_euler[0], s_rk4[0] = s_0, s_0

# Интегрирование системы
for i in range(n_steps):
    s_euler[i+1] = explicit_euler_step(s_euler[i], dt, d_linear)
    s_rk4[i+1] = rk4_step(s_rk4[i], dt, d_linear)

# Отсечение точек ниже уровня земли
valid_euler = s_euler[:, 1] >= 0
valid_rk4 = s_rk4[:, 1] >= 0

x_e, y_e = s_euler[valid_euler, 0], s_euler[valid_euler, 1]
x_rk, y_rk = s_rk4[valid_rk4, 0], s_rk4[valid_rk4, 1]

t_an = np.linspace(0, len(x_rk)*dt, 200)
x_an, y_an = analytical_solution(t_an)

# Энергетический анализ для RK4
v_sq = s_rk4[valid_rk4, 2]**2 + s_rk4[valid_rk4, 3]**2
E_k = 0.5 * m * v_sq
E_p = m * g * y_rk
E_full = E_k + E_p
t_energy = np.linspace(0, len(E_k)*dt, len(E_k))

# Визуализация результатов
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(x_an, y_an, 'k-', lw=2, label='Аналитическое решение')
ax1.plot(x_e, y_e, 'b--', lw=2, label='Эйлер (явный)')
ax1.plot(x_rk, y_rk, 'r-.', lw=2, label='Рунге-Кутта 4')
ax1.set(xlabel='X, м', ylabel='Y, м', title='Сравнение численных траекторий с аналитической')
ax1.grid(True, alpha=0.5)
ax1.legend()
ax1.set_ylim(bottom=0)

ax2.plot(t_energy, E_k, 'b-', label='Кинетическая энергия ($E_k$)')
ax2.plot(t_energy, E_p, 'g-', label='Потенциальная энергия ($E_p$)')
ax2.plot(t_energy, E_full, 'r-', lw=2, label='Полная механическая энергия ($E$)')
ax2.set(xlabel='Время $t$, с', ylabel='Энергия, Дж', title='Динамика энергии системы (метод RK4)')
ax2.grid(True, alpha=0.5)
ax2.legend()

plt.tight_layout()
plt.show()

## 🟡 Продвинутый уровень: Квадратичное сопротивление среды

Для квадратичного закона сопротивления система ОДУ модифицируется:
$$ \frac{d\vec{v}}{dt} = \vec{g} - k |\vec{v}|\vec{v} $$

In [ ]:
def simulate_trajectory(alpha, k, step_dt=0.01):
    vx, vy = v0 * np.cos(alpha), v0 * np.sin(alpha)
    s = np.array([0.0, 0.0, vx, vy])
    trajectory = [s]
    
    def d_quad(s_vec):
        x, y, vx, vy = s_vec
        v_mag = np.hypot(vx, vy)
        return np.array([vx, vy, -k * v_mag * vx, -g - k * v_mag * vy])
    
    while s[1] >= 0 or len(trajectory) == 1:
        s = rk4_step(s, step_dt, d_quad)
        trajectory.append(s)
        
    trajectory = np.array(trajectory)
    
    # Линейная интерполяция для точного момента пересечения поверхности земли
    s_last = trajectory[-1]
    s_prev = trajectory[-2]
    fraction = s_prev[1] / (s_prev[1] - s_last[1])
    x_land = s_prev[0] + fraction * (s_last[0] - s_prev[0])
    
    trajectory[-1, 0] = x_land
    trajectory[-1, 1] = 0.0
    return trajectory, x_land

# Поиск оптимального угла запуска
k_values = [0.0, 0.001, 0.005, 0.01]
angles = np.linspace(np.radians(0), np.radians(90), 601)

fig, ax = plt.subplots(figsize=(10, 6))

for k in k_values:
    ranges = [simulate_trajectory(a, k)[1] for a in angles]
    opt_idx = np.argmax(ranges)
    opt_angle = angles[opt_idx]
    max_range = ranges[opt_idx]
    
    ax.plot(np.degrees(angles), ranges, lw=2, label=f'k = {k}')
    ax.plot(np.degrees(opt_angle), max_range, 'ko')
    #if k > 0:
    ax.annotate(f"{np.degrees(opt_angle):.1f}°", 
                (np.degrees(opt_angle), max_range), 
                textcoords="offset points", xytext=(0,10), ha='center')

ax.set(title='Зависимость дальности полета от начального угла запуска',
       xlabel=r'Угол запуска $\alpha$, градусы',
       ylabel='Дальность полета $L$, м')
ax.grid(True, alpha=0.5)
ax.legend()
plt.show()

**Ответ на контрольный вопрос (влияние сопротивления на характеристики полета):**
Наличие квадратичного сопротивления среды необратимо нарушает симметрию параболической траектории. Восходящая ветвь становится более пологой, а нисходящая — крутой из-за кумулятивного падения горизонтальной составляющей скорости. Оптимальный угол запуска, обеспечивающий максимальную дальность, смещается со стандартных $45^{\circ}$ в сторону меньших углов. Запуск под меньшим углом сокращает время пребывания тела в воздухе, тем самым минимизируя потери энергии на преодоление диссипативных сил.

### Динамика полета и воздействие сил (Анимация 1)
Визуализация вектора скорости и векторов сил (гравитация и сопротивление).

In [ ]:
alpha_anim = np.radians(40)
k_anim = 0.005
traj_single, _ = simulate_trajectory(alpha_anim, k_anim, step_dt=0.05)

fig_single, ax_single = plt.subplots(figsize=(10, 5))
ax_single.set_xlim(0, np.max(traj_single[:, 0]) * 1.1)
ax_single.set_ylim(0, np.max(traj_single[:, 1]) * 1.2)
ax_single.set_aspect('equal')
ax_single.set_title("Векторы скорости и сил при квадратичном сопротивлении")
ax_single.grid(True, alpha=0.5)

line_traj, = ax_single.plot([], [], 'b--', lw=1, alpha=0.7)
point_body, = ax_single.plot([], [], 'ro', ms=6)
quiver_v = ax_single.quiver(0, 0, 0, 0, color='g', scale=250, width=0.003, headwidth=5, label='Скорость')
quiver_g = ax_single.quiver(0, 0, 0, 0, color='k', scale=50, width=0.003, headwidth=5, label='Гравитация')
quiver_f = ax_single.quiver(0, 0, 0, 0, color='r', scale=50, width=0.003, headwidth=5, label='Сопротивление')
ax_single.legend(loc='upper right')

plt.close() # Блокируем статичный вывод для корректного рендеринга анимации

def update_single(frame):
    s = traj_single[frame]
    x, y, vx, vy = s
    v_mag = np.hypot(vx, vy)
    
    line_traj.set_data(traj_single[:frame+1, 0], traj_single[:frame+1, 1])
    point_body.set_data([x], [y])
    
    quiver_v.set_offsets(np.c_[x, y])
    quiver_v.set_UVC(vx, vy)
    
    quiver_g.set_offsets(np.c_[x, y])
    quiver_g.set_UVC(0, -g)
    
    quiver_f.set_offsets(np.c_[x, y])
    quiver_f.set_UVC(-k_anim * v_mag * vx, -k_anim * v_mag * vy)
    return line_traj, point_body, quiver_v, quiver_g, quiver_f

ani_single = animation.FuncAnimation(fig_single, update_single, frames=len(traj_single), interval=40, blit=False)
HTML(ani_single.to_jshtml())

### Запуск снарядов (Анимация 2)
Сравнение одновременного запуска 10 тел в идеальных условиях ($k=0$) и в вязкой среде ($k=0.005$). Шкала времени согласована для обеих панелей.

In [ ]:
plt.rcParams['animation.embed_limit'] = 50.0

angles_multi = np.linspace(np.radians(20), np.radians(70), 10)
trajs_nodrag = [simulate_trajectory(a, 0.0, step_dt=0.05)[0] for a in angles_multi]
trajs_drag = [simulate_trajectory(a, 0.005, step_dt=0.05)[0] for a in angles_multi]

max_frames = max(max(len(tr) for tr in trajs_nodrag), max(len(tr) for tr in trajs_drag))

k_demo = 0.005
fig_multi, (ax_nd, ax_d) = plt.subplots(2, 1, figsize=(14, 12))
titles = ['Без сопротивления ($k=0$)', f'С сопротивлением ($k={k_demo}$)']

# Настройка первого графика (без сопротивления)
ax_nd.set_xlim(0, 260)
ax_nd.set_ylim(0, 140)
ax_nd.set_aspect('equal')
ax_nd.axhline(0, color='gray', lw=2)
ax_nd.set_title('Без сопротивления ($k=0$)')
ax_nd.set_xlabel('X, м')
ax_nd.set_ylabel('Y, м')
ax_nd.grid(True, alpha=0.3)

# Настройка второго графика (с сопротивлением). 
# Подстраиваем оси под реальный размер новых траекторий
max_x_d = max(np.max(tr[:, 0]) for tr in trajs_drag) * 1.05
max_y_d = max(np.max(tr[:, 1]) for tr in trajs_drag) * 1.15

ax_d.set_xlim(0, max_x_d)
ax_d.set_ylim(0, max_y_d)
ax_d.set_aspect('equal')
ax_d.axhline(0, color='gray', lw=2)
ax_d.set_title(f'С сопротивлением ($k={k_demo}$)')
ax_d.set_xlabel('X, м')
ax_d.set_ylabel('Y, м')
ax_d.grid(True, alpha=0.3)

points_nd = [ax_nd.plot([], [], 'o', ms=5)[0] for _ in range(10)]
trails_nd = [ax_nd.plot([], [], '-', lw=1.5, alpha=0.6)[0] for _ in range(10)]

points_d = [ax_d.plot([], [], 'o', ms=5)[0] for _ in range(10)]
trails_d = [ax_d.plot([], [], '-', lw=1.5, alpha=0.6)[0] for _ in range(10)]

plt.tight_layout()
plt.close()

def update_multi(frame):
    for i in range(10):
        # Вакуум
        idx_nd = min(frame, len(trajs_nodrag[i]) - 1)
        points_nd[i].set_data([trajs_nodrag[i][idx_nd, 0]], [trajs_nodrag[i][idx_nd, 1]])
        trails_nd[i].set_data(trajs_nodrag[i][:idx_nd+1, 0], trajs_nodrag[i][:idx_nd+1, 1])
        
        # Среда с сопротивлением
        idx_d = min(frame, len(trajs_drag[i]) - 1)
        points_d[i].set_data([trajs_drag[i][idx_d, 0]], [trajs_drag[i][idx_d, 1]])
        trails_d[i].set_data(trajs_drag[i][:idx_d+1, 0], trajs_drag[i][:idx_d+1, 1])
        
    return points_nd + trails_nd + points_d + trails_d

ani_multi = animation.FuncAnimation(fig_multi, update_multi, frames=max_frames, interval=30, blit=False)
HTML(ani_multi.to_jshtml())